In [89]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [90]:
words = open('names.txt','r').read().splitlines()

In [91]:
chars = sorted(list(set(''.join(words))))

In [128]:
# instead of the predicting the next letter wrt to the current letter we take the context of more than one letter 
#this is the block size
#X,Y is the dataset where X has the context and Y has the output letter
block_size = 3
def build_dataset(words):
    X , Y = [] , []
    for word in words:
        context = [0]*block_size
        for ch in word + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xtest,Ytest = build_dataset(words[n2:])

In [139]:
# need to embed the charecters into a vector of some dimension here 2
# randomly generate here and using the loss function and the backpropagation we can find the best embeddings
num_dims = 2
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,num_dims),generator = g)
embd = C[X]
W1 = torch.randn((6,300),generator = g)
b1 = torch.randn(300,generator = g)
W2 = torch.randn((300,27),generator = g)
b2 = torch.randn((27),generator = g)
#params
params = [C,W1,b1,W2,b2]
params
for p in params:
    p.requires_grad = True

In [ ]:
#training
for i in range(100):
    #forward pass
    embd = C[Xtr]
    l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
    l2 = l1 @ W2 + b2
    loss = F.cross_entropy(l2,Ytr)
    print(loss.item())
    
    #backward pass
    for p in params:
        p.grad = None
    loss.backward()

    #update
    for p in params:
        p.data -= 0.1*p.grad
    

In [150]:
#instead of calculating the loss on all of them we can use a mini batch for training
for i in range(30000):
    #forward pass
    ix = torch.randint(0,Xtr.shape[0],(32,))
    embd = C[Xtr[ix]]
    l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
    l2 = l1 @ W2 + b2
    loss = F.cross_entropy(l2,Ytr[ix])
    # print(loss.item())
    
    #backward pass
    for p in params:
        p.grad = None
    loss.backward()

    #update
    for p in params:
        p.data -= 0.01*p.grad
print(loss.item())

2.367236852645874


In [151]:
#total loss
embd = C[Xdev]
l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
l2 = l1 @ W2 + b2
loss = F.cross_entropy(l2,Ydev)
loss

tensor(2.2691, grad_fn=<NllLossBackward0>)

In [152]:
embd = C[Xtest]
l1 = torch.tanh(embd.view(embd.shape[0],block_size * num_dims) @ W1 + b1)
l2 = l1 @ W2 + b2
loss = F.cross_entropy(l2,Ytest)
loss

tensor(2.2719, grad_fn=<NllLossBackward0>)

In [171]:
#using the model now
for _ in range(20):
    out = []
    context = [0]*block_size
    while True:
        embd = C[torch.tensor([context])]
        l1 = torch.tanh(embd.view(1,-1) @ W1 + b1)
        l2 = l1 @ W2 + b2
        probs = F.softmax(l2,dim=1)
        ix = torch.multinomial(probs,num_samples =1).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(ix)
    print(''.join(itos[i] for i in out))
        

xelisetsy
tan
gica
yad
tha
poshumrirz
cozina
byus
gumias
azar
dair
gelyss
chishin
xma
anfel
haw
ameyela
aphel
jaily
meyta
